# Upstream-downstream split of the 1,083 basins
Input: `data/Lev08_cells.csv`, the Level-08 cells of each Level-07 basin (HydroBASINS / BasinATLAS v10 attributes).
Output: `split_output/`. T = 600 m is the run used in the paper; the rule is described in the README.

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
T_LIST = [300, 400, 500, 600, 700, 800]      # relief threshold (m)

In [ ]:
def wpct(x, w, q):                           # area-weighted percentile
    i = np.argsort(x)
    x, w = x[i], w[i]
    return float(np.interp(q, np.cumsum(w) / w.sum(), x))


def pre_label(g, T, rule):
    a = g.SUB_AREA.to_numpy(float)
    e = g.ele_mt_sav.to_numpy(float)
    d = g.PFAF_ID.to_numpy() % 10            # last Pfafstetter digit
    up, dn = d >= 6, (d >= 1) & (d <= 4)
    dE = wpct(e, a, .9) - wpct(e, a, .1)     # internal relief
    if rule == "AREA50":
        return None, "area50", dE
    if rule == "ELEVDEF" or dE >= T:
        return e >= wpct(e, a, .5), "elevdef" if rule else "elevation", dE
    if up.any() and dn.any():              
        return up | ((d == 5) & (abs(e - e[up].mean()) < abs(e - e[dn].mean()))), "topology_pfaf", dE
    s = g.DIST_SINK.to_numpy(float)
    return s >= wpct(s, a, .5), "topology_dist", dE


def cut(g, target):
    ids, a = g.HYBAS_ID_08.tolist(), g.SUB_AREA.to_numpy(float)
    area, kids, outlets = dict(zip(ids, a.tolist())), defaultdict(list), []
    for h, nd in zip(ids, g.NEXT_DOWN.tolist()):
        (kids[nd] if nd in area else outlets).append(h)

    sub = {}
    def drain(h):                          
        sub[h] = area[h]
        for c in kids.get(h, []):
            sub[h] += drain(c)
        return sub[h]
    for o in outlets:
        drain(o)

    stem = [max(outlets, key=sub.get)]      
    while kids.get(stem[-1]):
        stem.append(max(kids[stem[-1]], key=sub.get))
    tot = a.sum()
    todo, up = [min(stem, key=lambda h: abs(sub[h] / tot - target))], set()
    while todo:
        h = todo.pop()
        up.add(h)
        todo += kids.get(h, [])
    return np.isin(ids, list(up)), len(outlets)


def split(g, T=600, rule=None):
    a = g.SUB_AREA.to_numpy(float)
    pre, axis, dE = pre_label(g, T, rule)
    f0 = a[pre].sum() / a.sum() if pre is not None else np.nan
    up, n_out = cut(g, f0 if 0 < f0 < 1 else 0.5)
    return up, dict(PFAF_ID_07=g.PFAF_ID_07.iat[0], n_children=len(g), delta_E_m=round(dE, 1), axis=axis,
                    f_up_area=round(float(a[up].sum() / a.sum()), 3), multi_outlet=n_out > 1)

In [ ]:
cells = pd.read_csv(ROOT / "data/Lev08_cells.csv")
runs = [(f"T{t}", t, None) for t in T_LIST] + [("ELEVDEF", 600, "ELEVDEF"), ("AREA50", 600, "AREA50")]
lab, summ = cells[["HYBAS_ID_08", "PFAF_ID_07"]].copy(), {}
for name, T, rule in runs:
    up, rows = np.zeros(len(cells), bool), []
    for _, g in cells.groupby("PFAF_ID_07", sort=False):
        up[g.index], row = split(g, T, rule)
        rows.append(row)
    lab[f"s_{name}"] = np.where(up, "Upstream", "Downstream")
    summ[name] = pd.DataFrame(rows)

s = summ["T600"]
s["relief_class"] = np.select([s.delta_E_m >= 600, s.delta_E_m >= 300], ["high", "mixed"], "low")
s["split_degenerate"] = s.f_up_area.isin([0, 1])          
out = ROOT / "split_output"
out.mkdir(exist_ok=True)
lab.to_csv(out / "Lev08_split_thresholds.csv", index=False)
cells.assign(Stream=lab.s_T600).merge(s[["PFAF_ID_07", "axis", 
                                         "relief_class", "delta_E_m"]]).to_csv(out / "Lev08_split.csv", index=False)
s.to_csv(out / "basin_split_summary.csv", index=False)


In [ ]:
# same as the published tables?
pub = pd.read_csv(ROOT / "data/Lev08_split_thresholds.csv")
ps = pd.read_csv(ROOT / "data/basin_split_summary.csv")

In [ ]:
ps